In [2]:
import sys
sys.path.append('..')

from root import ROOT
from data.fwf_dataset import FwfDataset

from omegaconf import OmegaConf
import os
import json
import numpy as np
import pandas as pd

# assemble config
# cfg = OmegaConf.load(os.path.join(ROOT,'config/default.yaml'))
# cfg.general.root =  ROOT
# cfg = OmegaConf.merge(cfg, OmegaConf.load(cfg.data.split))
# with open(os.path.join(cfg.data.dataset_root, 'class_dict.json'),'r') as f:
#     cfg = OmegaConf.merge(cfg, OmegaConf.create({'data':{'label_schema':json.load(f)}}))


# print(OmegaConf.to_yaml(cfg))

In [3]:
from plyfile import PlyData, PlyElement
from utils.metrics import simple_metrics

# validate with 3-rd party libraries
import torch
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassAccuracy
from sklearn.metrics import jaccard_score

from glob import glob

batches = [
    'batch_5b_std/run_0',
    'batch_5b_std/run_1',
    'batch_5b_std/run_2',
    'batch_5b_std/run_3',
]

overall_results = list()
classwise_results = list()

for batch in batches:
    print(batch)

    # accumulate results
    all_metrics = dict()
    mious = dict()
    maccs = dict()
    cfg = None
    for exp_fp in glob(os.path.join(ROOT,f'exp/{batch}/*_v*')):
        exp_name = os.path.basename(exp_fp)
        cfg = OmegaConf.load(os.path.join(exp_fp, 'config.yaml'))
        num_classes = len(cfg.data.label_schema[cfg.data.label_names[0]])
        print(exp_name)
        # load point clouds

        gts = []
        preds = []
        for pcd_fp in glob(os.path.join(exp_fp,'test/*.ply')):
            # aggregate labels from all point clodus
            pcd_name = os.path.basename(pcd_fp)
            print(f'\t{pcd_name}')
            pcd = pd.DataFrame.from_dict(PlyData.read(pcd_fp).elements[0].data)
            gt = pcd['gt_labels_3'].to_numpy().astype(np.uint8)
            pred = pcd['pr_labels_3'].to_numpy().astype(np.uint8)
            gts.append(gt)
            preds.append(pred)
            

        # check metrics
        # miou_own = simple_metrics(gt, pred, num_classes)['miou']
        # miou_skl = jaccard_score(gt, pred, average='macro')
        gt = np.concatenate(gts)
        pred = np.concatenate(preds)
        miou_metric = MulticlassJaccardIndex(num_classes=num_classes, average=None)
        iou_pytorch = miou_metric(torch.tensor(gt), torch.tensor(pred)).numpy()
        # print(f"\t\t{miou_own=}\n\t\t{miou_skl=}\n\t\t{miou_pytorch=}")
        all_metrics[exp_name] = iou_pytorch

        miou = MulticlassJaccardIndex(num_classes=num_classes, average='macro')
        macc = MulticlassAccuracy(num_classes=num_classes,average='macro')
        mious[exp_name] = miou(torch.tensor(gt), torch.tensor(pred)).numpy() # N,
        maccs[exp_name] = macc(torch.tensor(gt), torch.tensor(pred)).numpy() # N,
            
    # create dataframe for overall results
    names = pd.Series(mious.keys())
    df_new = pd.DataFrame({'names': names,
                        'mious': [mious[n]*100 for n in names],
                        'maccs': [maccs[n]*100 for n in names]
                        })
    df_new['names'] = df_new['names'].apply(lambda x: int(x.split('_')[-1].replace('v',''))-500)
    df_new = df_new.sort_values(by='names').reset_index(drop=True)
    overall_results.append(df_new)

    # create dataframe for class-wise results
    cat_names = list(cfg.data.label_schema['labels_3'].keys())
    all_metrics_df = pd.DataFrame(columns=['exp_name']+ cat_names)
    all_metrics_df

    for exp_name, metrics in all_metrics.items():
        # all_metrics_df['exp_name'] = exp_name
        # all_metrics_df[cat_names] = metrics
        exp_metrics = pd.DataFrame([{**{'exp_name': exp_name.split('_')[-1]}, **{n: m*100 for n, m in zip(cat_names, metrics)}}])
        all_metrics_df = pd.concat([all_metrics_df, exp_metrics], axis=0, ignore_index=True)
        # exp_metrics[exp_metrics] = exp_name

    all_metrics_df.sort_values(by='exp_name')
    classwise_results.append(all_metrics_df)
    






batch_5_std/run_0
2024-12-25_09-06-29_v500
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-25_11-10-19_v501
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-25_13-16-29_v502
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-25_15-25-48_v503
	2023-08-28_FW_EingangBauing.FwfP

/tmp/ipykernel_76962/2979612472.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_metrics_df = pd.concat([all_metrics_df, exp_metrics], axis=0, ignore_index=True)


batch_5_std/run_1
2024-12-26_11-11-42_v500
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-26_13-14-38_v501
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-26_15-20-24_v502
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-26_17-28-41_v503
	2023-08-28_FW_EingangBauing.FwfP

/tmp/ipykernel_76962/2979612472.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_metrics_df = pd.concat([all_metrics_df, exp_metrics], axis=0, ignore_index=True)


batch_5_std/run_2
2024-12-27_11-17-13_v500
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-27_13-20-36_v501
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-27_15-27-12_v502
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-27_17-36-22_v503
	2023-08-28_FW_EingangBauing.FwfP

/tmp/ipykernel_76962/2979612472.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_metrics_df = pd.concat([all_metrics_df, exp_metrics], axis=0, ignore_index=True)


batch_5_std/run_3
2024-12-30_08-55-30_v500
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-30_10-59-28_v501
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-30_13-05-48_v502
	2023-08-28_FW_EingangBauing.FwfProj--defaultBbox.ply
	2024-04-05_FW_Westbahnhof_01.FwfProj--defaultBbox.ply
	2024-07-31_FW_Bruecke_Koenigstr.FwfProj--bboxId=001.ply
	2024-08-02_FW_Bruecke_Kasinostrasse.FwfProj--bboxId=000.ply
	2024-07-31_FW_Bruecke_Turmstr.FwfProj--defaultBbox.ply
2024-12-30_15-14-23_v503
	2023-08-28_FW_EingangBauing.FwfP

/tmp/ipykernel_76962/2979612472.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_metrics_df = pd.concat([all_metrics_df, exp_metrics], axis=0, ignore_index=True)


In [ ]:
means
 \begin{tabular}{lrrr}
\toprule
 & names & mious & maccs \\
\midrule
0 & 0 & 2.12 & 6.87 \\
1 & 1 & 12.37 & 22.66 \\
2 & 2 & 20.59 & 33.07 \\
3 & 3 & 23.69 & 35.96 \\
4 & 4 & 20.22 & 32.26 \\
5 & 5 & 17.47 & 30.17 \\
6 & 6 & 29.60 & 41.52 \\
7 & 7 & 0.55 & 4.80 \\
8 & 8 & 15.57 & 25.19 \\
\bottomrule
\end{tabular}

std
 \begin{tabular}{lrrr}
\toprule
 & names & mious & maccs \\
\midrule
0 & 0 & 2.49 & 3.86 \\
1 & 1 & 0.75 & 0.81 \\
2 & 2 & 0.13 & 1.22 \\
3 & 3 & 0.46 & 1.83 \\
4 & 4 & 0.77 & 0.62 \\
5 & 5 & 1.12 & 1.61 \\
6 & 6 & 0.76 & 1.27 \\
7 & 7 & 0.33 & 0.65 \\
8 & 8 & 0.74 & 1.06 \\
\bottomrule
\end{tabular}


In [24]:
# means
overall_results_means = np.array([o.to_numpy() for o in overall_results]).mean(axis=0)
overall_results_means = pd.DataFrame(overall_results_means, columns=overall_results[0].columns)
overall_results_means['names'] = overall_results[0]['names']
print('means\n',overall_results_means.to_latex(float_format=('%.2f')))
# stds
overall_results_std = np.array([o.to_numpy() for o in overall_results]).std(axis=0)
overall_results_std = pd.DataFrame(overall_results_std, columns=overall_results[0].columns)
overall_results_std['names'] = overall_results[0]['names']
print('std\n',overall_results_std.to_latex(float_format='%.2f'))


means
 \begin{tabular}{lrrr}
\toprule
 & names & mious & maccs \\
\midrule
0 & 0 & 2.12 & 6.87 \\
1 & 1 & 12.37 & 22.66 \\
2 & 2 & 20.59 & 33.07 \\
3 & 3 & 23.69 & 35.96 \\
4 & 4 & 20.22 & 32.26 \\
5 & 5 & 17.47 & 30.17 \\
6 & 6 & 29.60 & 41.52 \\
7 & 7 & 0.55 & 4.80 \\
8 & 8 & 15.57 & 25.19 \\
\bottomrule
\end{tabular}

std
 \begin{tabular}{lrrr}
\toprule
 & names & mious & maccs \\
\midrule
0 & 0 & 2.49 & 3.86 \\
1 & 1 & 0.75 & 0.81 \\
2 & 2 & 0.13 & 1.22 \\
3 & 3 & 0.46 & 1.83 \\
4 & 4 & 0.77 & 0.62 \\
5 & 5 & 1.12 & 1.61 \\
6 & 6 & 0.76 & 1.27 \\
7 & 7 & 0.33 & 0.65 \\
8 & 8 & 0.74 & 1.06 \\
\bottomrule
\end{tabular}



In [45]:
classwise_results[0]

,exp_name,_unspecified,asphalt,brick,cable,concrete,marking,mesh,metal,naturalStone,poster,treeTrunk,vegetation
0,0,0.000082,36.236462,0.000138,0.175074,9.306239,0.000000,0.001365,0.000502,0.000000,3.614074,0.613220,26.984102
1,1,3.276158,39.446101,0.000000,0.683084,35.110879,2.576210,7.162827,0.787228,0.000000,11.688302,2.668024,49.123010
2,2,8.184817,22.811115,0.006469,3.018164,45.418349,1.166886,64.574742,9.329186,0.000564,26.226640,7.859553,60.325676
3,3,4.986502,34.347647,0.005505,7.066491,47.944868,17.259790,58.130580,35.671285,0.000000,27.091914,5.268533,55.464453
4,4,8.180833,45.055664,0.121404,5.868899,43.381950,10.097429,19.494109,36.709169,0.000000,18.071534,4.146067,56.392801
5,5,5.996380,30.379298,0.010443,0.714519,37.637040,3.507773,21.414238,20.341843,0.000000,17.744069,4.739897,49.447542
6,6,8.848722,53.318936,0.008777,23.975162,46.976775,18.005158,60.882580,39.821875,0.000000,33.794600,9.992167,67.146951
7,7,0.000000,1.021316,0.000000,0.000000,4.116235,0.000000,0.000000,0.000000,0.000000,1.743103,0.000000,0.009475
8,8,3.385923,50.776494,0.000000,9.424482,35.726374,4.484503,2.238222,2.586381,0.000000,17.516287,1.548334,65.382892


In [35]:
for c in classwise_results:
    c['exp_name'] = c['exp_name'].apply(lambda x: int(x.split('_')[-1].replace('v',''))-500)
    c = c.sort_values(by='exp_name').reset_index(drop=True)

    # overall_results.append(df_new)

In [62]:
np.set_printoptions(suppress=True)
classwise_results_means

array([[ 0.        ,  0.0000204 , 10.63473277,  0.00003441,  0.04376854,
         5.64748207,  0.        ,  0.0003413 ,  0.0001255 ,  0.        ,
         2.21324493,  0.15330488,  6.76542852],
       [ 1.        ,  1.38902192, 39.98868912,  0.        ,  1.52110481,
        33.72419477,  2.22326798,  6.07904792,  0.68843047,  0.00001175,
        10.24199259,  2.936571  , 49.59819838],
       [ 2.        ,  7.91595727, 22.72898518,  0.00361317,  4.3666183 ,
        43.61128062,  2.51861578, 62.66688406,  9.50509738,  0.00014106,
        24.80256595,  6.74945368, 62.19775379],
       [ 3.        ,  4.83993888, 34.97897238,  0.0133528 ,  3.09816811,
        45.93595564, 15.30061848, 57.83363432, 35.67870185,  0.00005877,
        23.89985695,  6.21617762, 56.52057528],
       [ 4.        ,  7.55443238, 44.82359663,  0.08377626,  6.69709854,
        42.72064418,  8.39850195, 17.73390621, 34.32243615,  0.        ,
        20.18589042,  5.15452875, 54.97796088],
       [ 5.        ,  6.574999

In [63]:
classwise_results[0].columns

Index(['exp_name', '_unspecified', 'asphalt', 'brick', 'cable', 'concrete',
       'marking', 'mesh', 'metal', 'naturalStone', 'poster', 'treeTrunk',
       'vegetation'],
      dtype='object')

In [64]:
classwise_results
# overall_results_means
classwise_results_means = np.array([o.to_numpy() for o in classwise_results]).mean(axis=0)

classwise_results_means = pd.DataFrame(classwise_results_means, columns=classwise_results[0].columns)
classwise_results_means['exp_name'] = classwise_results[0]['exp_name']
print('std\n',classwise_results_means.to_latex(float_format='%.2f'))

std
 \begin{tabular}{lrrrrrrrrrrrrr}
\toprule
 & exp_name & _unspecified & asphalt & brick & cable & concrete & marking & mesh & metal & naturalStone & poster & treeTrunk & vegetation \\
\midrule
0 & 0 & 0.00 & 10.63 & 0.00 & 0.04 & 5.65 & 0.00 & 0.00 & 0.00 & 0.00 & 2.21 & 0.15 & 6.77 \\
1 & 1 & 1.39 & 39.99 & 0.00 & 1.52 & 33.72 & 2.22 & 6.08 & 0.69 & 0.00 & 10.24 & 2.94 & 49.60 \\
2 & 2 & 7.92 & 22.73 & 0.00 & 4.37 & 43.61 & 2.52 & 62.67 & 9.51 & 0.00 & 24.80 & 6.75 & 62.20 \\
3 & 3 & 4.84 & 34.98 & 0.01 & 3.10 & 45.94 & 15.30 & 57.83 & 35.68 & 0.00 & 23.90 & 6.22 & 56.52 \\
4 & 4 & 7.55 & 44.82 & 0.08 & 6.70 & 42.72 & 8.40 & 17.73 & 34.32 & 0.00 & 20.19 & 5.15 & 54.98 \\
5 & 5 & 6.57 & 34.47 & 0.01 & 3.53 & 40.41 & 4.42 & 25.69 & 19.30 & 0.00 & 19.36 & 5.07 & 50.78 \\
6 & 6 & 9.11 & 48.13 & 0.06 & 25.91 & 48.11 & 18.19 & 62.96 & 40.62 & 0.00 & 23.57 & 9.97 & 68.62 \\
7 & 7 & 0.00 & 0.93 & 0.00 & 0.00 & 3.89 & 0.00 & 0.00 & 0.00 & 0.00 & 1.74 & 0.00 & 0.02 \\
8 & 8 & 4.94 & 51.30 & 

In [ ]:
0 & 0 & 0.00 & 10.63 & 0.00 & 0.04 & 5.65 & 0.00 & 0.00 & 0.00 & 0.00 & 2.21 & 0.15 & 6.77 \\
1 & 1 & 1.39 & 39.99 & 0.00 & 1.52 & 33.72 & 2.22 & 6.08 & 0.69 & 0.00 & 10.24 & 2.94 & 49.60 \\
2 & 2 & 7.92 & 22.73 & 0.00 & 4.37 & 43.61 & 2.52 & 62.67 & 9.51 & 0.00 & 24.80 & 6.75 & 62.20 \\
3 & 3 & 4.84 & 34.98 & 0.01 & 3.10 & 45.94 & 15.30 & 57.83 & 35.68 & 0.00 & 23.90 & 6.22 & 56.52 \\
4 & 4 & 7.55 & 44.82 & 0.08 & 6.70 & 42.72 & 8.40 & 17.73 & 34.32 & 0.00 & 20.19 & 5.15 & 54.98 \\
5 & 5 & 6.57 & 34.47 & 0.01 & 3.53 & 40.41 & 4.42 & 25.69 & 19.30 & 0.00 & 19.36 & 5.07 & 50.78 \\
6 & 6 & 9.11 & 48.13 & 0.06 & 25.91 & 48.11 & 18.19 & 62.96 & 40.62 & 0.00 & 23.57 & 9.97 & 68.62 \\
7 & 7 & 0.00 & 0.93 & 0.00 & 0.00 & 3.89 & 0.00 & 0.00 & 0.00 & 0.00 & 1.74 & 0.00 & 0.02 \\
8 & 8 & 4.94 & 51.30 & 0.00 & 5.16 & 32.88 & 2.12 & 1.86 & 3.00 & 0.00 & 17.19 & 2.39 & 65.97 \\